#  Búsqueda de Imágenes Similares con Embeddings

## 1. Configuración (Setup)
En esta sección prepararemos nuestro entorno de trabajo. Necesitaremos librerías para el manejo de tensores (TensorFlow), extracción de características (TensorFlow Hub), y para la creación de nuestra interfaz gráfica interactiva (Gradio).

In [ ]:
# Instalación de librerías necesarias
!pip install -q tensorflow tensorflow-hub gradio pillow tqdm numpy

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import gradio as gr
from PIL import Image
import os
from tqdm import tqdm

# Variables Globales
IMAGE_FOLDER = "image_dataset" # Carpeta con imágenes de referencia
MODEL_URL = "https://tfhub.dev/google/tf2-preview/mobilenet_v2/feature_vector/4"

# Crear carpeta de dataset si no existe para evitar errores
os.makedirs(IMAGE_FOLDER, exist_ok=True)
print("Librerías importadas y entorno configurado correctamente.")

## 2. Introducción Teórica

Los **Embeddings de Imágenes** son representaciones vectoriales densas (arrays de números) que capturan las características semánticas de una imagen. En lugar de comparar píxel por píxel, comparamos estos vectores.

Para medir qué tan similares son dos imágenes, utilizamos el **Producto Escalar (Dot Product)** o la **Similitud del Coseno**. Si tenemos dos vectores de embeddings $u$ y $v$, el producto escalar se define como:

$$ Similitud(u, v) = u \cdot v = \sum_{i=1}^{n} u_i v_i $$

Un valor mayor indica que las imágenes son más similares en el espacio de características extraído por la red neuronal (en este caso, MobileNet V2).

## 3. Implementación (Paso a Paso)

### 3.1. Carga del Modelo Preentrenado
Cargaremos **MobileNet V2** desde TensorFlow Hub. Este modelo ya fue entrenado con millones de imágenes y lo usaremos solo para extraer el vector de características, sin la capa final de clasificación.

In [ ]:
# Cargar modelo de extracción de características
print("Cargando modelo desde TF Hub...")
model = hub.load(MODEL_URL)
print("Modelo cargado con éxito.")

### 3.2. Función de Preprocesamiento y Extracción
Esta función toma una imagen, la redimensiona al tamaño esperado por MobileNet V2 (224x224), normaliza sus píxeles a valores entre 0 y 1, y la pasa por el modelo para obtener su embedding.

In [ ]:
def get_image_embedding(image):
    # Preprocesamiento de la imagen
    image = image.convert("RGB").resize((224, 224))
    image = np.array(image) / 255.0  # Normalizar valores
    image = tf.convert_to_tensor(image, dtype=tf.float32)
    image = tf.expand_dims(image, axis=0)  # Agregar dimensión de batch (1, 224, 224, 3)

    # Extraer características
    embedding = model(image)
    return np.squeeze(embedding) # Eliminar la dimensión de batch

### 3.3. Construcción de la Base de Datos de Vectores
Procesamos todas las imágenes de nuestra carpeta local y guardamos sus embeddings en un diccionario para búsquedas rápidas.

In [ ]:
# Base de datos de imágenes y sus embeddings
image_database = {}

# Puedes agregar imágenes a la carpeta 'image_dataset' para que esto funcione
if os.path.exists(IMAGE_FOLDER) and len(os.listdir(IMAGE_FOLDER)) > 0:
    for image_name in tqdm(os.listdir(IMAGE_FOLDER), desc="Procesando imágenes de referencia"):
        image_path = os.path.join(IMAGE_FOLDER, image_name)
        try:
            image = Image.open(image_path)
            image_database[image_name] = get_image_embedding(image)
        except Exception as e:
            print(f"No se pudo procesar {image_name}: {e}")
else:
    print(f"La carpeta '{IMAGE_FOLDER}' está vacía. Añade imágenes para tener resultados de búsqueda.")

### 3.4. Lógica de Búsqueda de Similitud
Calculamos la similitud entre el embedding de la imagen de consulta (query) y todos los embeddings en nuestra base de datos.

In [ ]:
def find_similar_images(query_image, top_k=5):
    if not image_database:
        raise ValueError("La base de datos de imágenes está vacía.")

    query_embedding = get_image_embedding(query_image)
    similarities = {}

    for image_name, embedding in image_database.items():
        # Similitud del coseno para normalizar entre -1 y 1
        cos_sim = np.dot(query_embedding, embedding) / (np.linalg.norm(query_embedding) * np.linalg.norm(embedding))
        # Limitamos a 0 para que Gradio no tenga problemas con negativos y muestre de 0 a 100%
        similarities[image_name] = float(max(0.0, cos_sim))

    # Ordenar por similitud descendente y tomar el top_k
    sorted_similarities = dict(sorted(similarities.items(), key=lambda item: item[1], reverse=True)[:top_k])

    return sorted_similarities

def search_images(image):
    try:
        similarities = find_similar_images(image, top_k=5)
        return similarities
    except Exception as e:
        return {f"Error: {str(e)}": 0.0}

## 4. Visualización e Interactividad

Utilizaremos **Gradio** para lanzar una aplicación web integrada en el cuaderno. Simplemente sube una imagen y el sistema te devolverá la más parecida de tu carpeta `image_dataset`.

*(Puedes usar los @param para experimentar cambiando el título o descripción de la UI rápida)*

In [ ]:
# @title Configuración de la Interfaz UI
ui_title = "Imagenes Similares (MobileNetV2)"
ui_description = "Sube una imagen y encuentra el top 5 de imágenes más parecidas en tu base de datos local."

iface = gr.Interface(
    fn=search_images,
    inputs=gr.Image(type="pil", label="Imagen de Consulta"),
    outputs=gr.Label(num_top_classes=5, label="Top 5 Archivos Similares"),
    title=ui_title,
    description=ui_description
)

# Validar que la base de datos no esté vacía antes de lanzar la interfaz
import os

if not os.path.exists(IMAGE_FOLDER) or len(os.listdir(IMAGE_FOLDER)) == 0:
    print(f"\u26A0\uFE0F Error: La carpeta '{IMAGE_FOLDER}' está vacía o no existe. Por favor, añade imágenes antes de iniciar la aplicación.")
else:
    # Cierra instancias previas para evitar errores de Asyncio en Colab
    gr.close_all()
    # Lanzar interfaz (debug=True permite ver errores en la celda)
    iface.launch(debug=True, share=True)
